# Stage 6 — Vector Store

**Project:** ResearchMate — Research Paper RAG Chatbot
**Goal of this notebook:** Build a FAISS vector store containing all 20,971 embedded document chunks, confirm similarity search works end-to-end, and save the index to disk for reuse in later stages.

**Before running:** upload `train_prepared.csv` (from Stage 2) to this Colab session.

## Cell 1 — Install packages

`faiss-cpu` is the actual FAISS library. `langchain-community` provides LangChain's wrapper around FAISS so it works smoothly with our `Document` objects and embedding model.

In [1]:
!pip install -q faiss-cpu langchain-community langchain-huggingface sentence-transformers langchain-core langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Cell 2 — Reload data and rebuild documents (same as Stages 3-5)

Self-contained setup: load the prepared CSV, rebuild the `Document` list, and split (still expected to be a 1:1 no-op, as confirmed in Stage 4).

In [2]:
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

df = pd.read_csv("train_prepared.csv")

documents = []
for _, row in df.iterrows():
    doc = Document(
        page_content=row["text"],
        metadata={
            "id": row["ID"],
            "title": row["TITLE"],
            "topics": row["topics"],
        }
    )
    documents.append(doc)

splitter = RecursiveCharacterTextSplitter(chunk_size=3500, chunk_overlap=200)
split_docs = splitter.split_documents(documents)

print("Total chunks to embed and store:", len(split_docs))

Total chunks to embed and store: 20971


## Cell 3 — Check whether Colab gave us a GPU

Embedding is much faster on a GPU. This just tells us what we're working with, so we know whether to expect the full-corpus embedding to take minutes or much longer. If this prints `False` and Cell 5's timing estimate looks too slow, go to **Runtime → Change runtime type → T4 GPU** and re-run from Cell 1.

In [3]:
import torch

gpu_available = torch.cuda.is_available()
print("GPU available:", gpu_available)
if gpu_available:
    print("Device name:", torch.cuda.get_device_name(0))

GPU available: True
Device name: Tesla T4


## Cell 4 — Load the embedding model

Same model as Stage 5: `sentence-transformers/all-MiniLM-L6-v2`, run locally.

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


## Cell 5 — Get a realistic timing estimate on a larger sample

Stage 5's 5-document timing test was misleading because model warm-up overhead dominated it. Here we time 500 real documents instead — large enough that warm-up cost gets averaged out, giving a much more trustworthy estimate for the full 20,971.

In [5]:
import time

sample_docs = split_docs[:500]
sample_texts = [doc.page_content for doc in sample_docs]

start = time.time()
_ = embedding_model.embed_documents(sample_texts)
elapsed = time.time() - start

rate = len(sample_docs) / elapsed
estimated_full_time = len(split_docs) / rate

print(f"Embedded {len(sample_docs)} documents in {elapsed:.1f} seconds ({rate:.1f} docs/sec)")
print(f"Estimated time for all {len(split_docs)} chunks: {estimated_full_time/60:.1f} minutes")

Embedded 500 documents in 2.0 seconds (254.8 docs/sec)
Estimated time for all 20971 chunks: 1.4 minutes


## Cell 6 — Build the FAISS vector store (embeds + indexes all documents in one call)

`FAISS.from_documents` does two things internally: it calls our embedding model on every document's `page_content`, and it builds the FAISS index structure over the resulting vectors — all while keeping each vector linked to its original `Document` (text + metadata). This is the one call that actually processes the full 20,971 chunks, so it may take a few minutes based on Cell 5's estimate.

In [6]:
from langchain_community.vectorstores import FAISS

start = time.time()
vectorstore = FAISS.from_documents(split_docs, embedding_model)
elapsed = time.time() - start

print(f"Vector store built in {elapsed/60:.1f} minutes")
print("Total vectors stored:", vectorstore.index.ntotal)

/tmp/ipykernel_584/2726166683.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Vector store built in 1.0 minutes
Total vectors stored: 20971


## Cell 7 — Test similarity search directly on the vector store

Before building a formal LangChain retriever (Stage 7), we can already query the vector store directly with `similarity_search`, to confirm the whole embed → index → search pipeline actually works end-to-end. We print each result's title, topics, and a short snippet of the abstract.

In [7]:
query = "What research has been done on graph neural networks?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"--- Result {i} ---")
    print("Title:", doc.metadata["title"])
    print("Topics:", doc.metadata["topics"])
    print("Snippet:", doc.page_content[:200], "...")
    print()

--- Result 1 ---
Title: Residual Gated Graph ConvNets
Topics: Computer Science, Statistics
Snippet: Residual Gated Graph ConvNets

  Graph-structured data such as social networks, functional brain networks,
gene regulatory networks, communications networks have brought the interest in
generalizing d ...

--- Result 2 ---
Title: Pitfalls of Graph Neural Network Evaluation
Topics: Computer Science
Snippet: Pitfalls of Graph Neural Network Evaluation

  Semi-supervised node classification in graphs is a fundamental problem in
graph mining, and the recently proposed graph neural networks (GNNs) have
achie ...

--- Result 3 ---
Title: Typed Graph Networks
Topics: Computer Science, Statistics
Snippet: Typed Graph Networks

  Recently, the deep learning community has given growing attention to neural
architectures engineered to learn problems in relational domains. Convolutional
Neural Networks empl ...



## Cell 8 — Save the vector store to disk (persistence)

`save_local` writes the FAISS index and its metadata to a folder, so we don't have to re-embed all 20,971 documents every time we reopen this project. We then zip that folder so it's easy to download from Colab and re-upload in Stage 7 onward.

In [8]:
import shutil

vectorstore.save_local("faiss_index")
shutil.make_archive("faiss_index", "zip", "faiss_index")

print("Saved vector store to 'faiss_index/' and zipped it as 'faiss_index.zip'")

Saved vector store to 'faiss_index/' and zipped it as 'faiss_index.zip'


## What to check after running this notebook

- **Cell 3:** note whether you have a GPU — relevant context if Cell 5's estimate looks slow.
- **Cell 5:** note the docs/sec rate and the estimated full-corpus time — much more trustworthy than Stage 5's 5-document estimate.
- **Cell 6:** confirm `vectorstore.index.ntotal` equals **20971** — every chunk got embedded and stored.
- **Cell 7:** look at the 3 retrieved results for the graph neural networks query — do they actually look relevant to the question?
- **Cell 8:** confirm `faiss_index.zip` was created — **download it** from Colab's file browser (left sidebar), since Stage 7 will need to re-load this same vector store instead of rebuilding it from scratch.

Paste back: the GPU check (Cell 3), the docs/sec rate and estimated time (Cell 5), the total vector count (Cell 6), and the 3 retrieved titles from Cell 7 — then we'll move to Stage 7 (Retriever).